# Wan 2.2 I2V A14B — deux vidéos « première image → dernière image » sur GPU T4 (Kaggle)

Ce notebook génère **V1.mp4** (E1 → E2) et **V2.mp4** (E2 → E3) avec Wan 2.2 I2V A14B quantifié en GGUF Q4_K_M, la LoRA de vitesse Lightning (lightx2v, 4 pas) et ComfyUI utilisé comme bibliothèque Python (aucune interface web, aucun widget).

**Avant de lancer (panneau de droite de Kaggle)**
1. *Accelerator* : **GPU T4 x2** (un seul T4 est utilisé ; P100 possible mais non testé).
2. *Internet* : **On** (nécessite un compte Kaggle vérifié par téléphone).
3. *Save Version* → **Save & Run All (Commit)**. Le notebook s'exécute de haut en bas sans intervention.

**Sorties** (dans `/kaggle/working`, onglet *Output*) : `V1.mp4`, `V2.mp4` (H.264, yuv420p, 480×832, 65 images à 16 i/s ≈ 4 s) et `journal.json` (paramètres, durées, versions des modèles, licences). Les modèles (~26 Gio) sont téléchargés **hors** de `/kaggle/working` (dans `/kaggle/temp`, `/kaggle/tmp` ou `/tmp`, celui qui a le plus d'espace libre).

**Étapes** : 1 environnement → 2 images d'entrée → 3 installation → 4 modèles → 5 ComfyUI + test SageAttention → 6 prompts et conditionnement première/dernière image → 7 passe *high noise* → 8 passe *low noise* → 9 décodage + MP4 → 10 journal.

La logique est réécrite (le notebook Colab d'Isi-dev qui a servi de référence n'a pas de licence) ; ses réglages rapides sont repris : 4 pas dont 2 sur le modèle *high noise*, euler / simple, CFG 1, shift 8, LoRA Lightning à 1,5, NAG sur le modèle *high noise*.

Licences : Wan 2.2 et fichiers de modèles dérivés Apache-2.0 ; ComfyUI GPL-3.0 ; ComfyUI-GGUF Apache-2.0 ; ComfyUI-KJNodes GPL-3.0 ; SageAttention 1.0.6 BSD-3-Clause. Détail dans `journal.json`.

In [ ]:
# =====================================================================
# CONFIG — tous les paramètres sont réunis ici (aucune autre cellule à modifier)
# =====================================================================
import os

# Un seul GPU est utilisé (ComfyUI n'exploite pas deux T4 en parallèle).
# Ces variables doivent être posées AVANT tout import de torch.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# --- Dossiers ---
OUT_DIR = "/kaggle/working"                     # exporté (limite 20 Go) : seulement V1.mp4, V2.mp4, journal.json
WORK_DIR_CANDIDATES = ["/kaggle/temp", "/kaggle/tmp", "/tmp"]  # on retient celui qui a le plus d'espace libre
WORK_SUBDIR = "wan22_flf2v"                     # ComfyUI, modèles et fichiers temporaires
DELETE_WORK_DIR_AT_END = False                  # True : supprime ComfyUI + modèles à la fin

# --- Images d'entrée (téléchargées au début ; arrêt avec message clair si une URL ne répond pas) ---
INPUT_IMAGES = {
    "E1": "https://mcp-tools-z-image-turbo.hf.space/--replicas/zpmc2/gradio_api/file=/tmp/gradio/ca77b3b6612688edf0b8eeddb58fa3da244e4c9f836df6381af2c45251d068c0/image.webp",
    "E2": "https://mcp-tools-flux-1-kontext-dev.hf.space/--replicas/dt994/gradio_api/file=/tmp/gradio/e5242ef3dd443e7ce4f391b6bf226a8f826b54bfa7aab6df72d2fe86c2482d28/image.webp",
    "E3": "https://mcp-tools-flux-1-kontext-dev.hf.space/--replicas/dt994/gradio_api/file=/tmp/gradio/a9990d987a667143a9de5b2f6b706249d17e5eb75b710e92b7ce00bdc513fa8b/image.webp",
}
ALLOW_LOCAL_IMAGE_FALLBACK = False   # True : si une URL échoue, utilise /kaggle/input/**/E1.(webp|png|jpg) (Dataset ajouté)

# --- Format vidéo ---
WIDTH, HEIGHT = 480, 832             # portrait 9:16 arrondi aux multiples de 16
FRAMES = 65                          # 4k+1 obligatoire ; 65 images = 17 images latentes
FPS = 16                             # 65 / 16 ≈ 4,06 s
SEED = 1101

# --- Échantillonnage (réglages rapides Lightning / lightx2v du notebook de référence) ---
STEPS = 4
HIGH_NOISE_STEPS = 2                 # pas 0→2 : modèle high noise ; pas 2→4 : modèle low noise
CFG = 1.0
SAMPLER = "euler"
SCHEDULER = "simple"
SHIFT_HIGH = 8.0
SHIFT_LOW = 8.0
LORA_STRENGTH_HIGH = 1.5
LORA_STRENGTH_LOW = 1.5
NAG_SCALE, NAG_ALPHA, NAG_TAU = 11.0, 0.25, 2.5   # NAG : fait agir le prompt négatif malgré CFG = 1 (high noise)
SAGE_ATTENTION = "auto"              # "auto" : SageAttention si le test GPU passe, sinon attention PyTorch ; "off"

# --- Encodage MP4 ---
X264_CRF = 16
X264_PRESET = "slow"

# --- Prompts ---
NEGATIVE_PROMPT = "text, letters, words, logo, watermark, label, brand, bottle, jerrycan, hands, people, cartoon, illustration, blurry, deformed metal, extra parts, bright background"
VIDEOS = [
    {"name": "V1", "start": "E1", "end": "E2",
     "prompt": "Slow continuous camera push-in from the three-quarter top view toward the open oil filler, then gently diving into the open valve cover. Golden motor oil flows over the camshaft lobes. Smooth steady cinematic motion, no cuts, realistic viscous oil. No text, no logo."},
    {"name": "V2", "start": "E2", "end": "E3",
     "prompt": "Slow continuous camera pull-back revealing the full engine cutaway. Golden oil runs down past the pistons onto the crankshaft. Smooth steady cinematic motion, no cuts, realistic viscous oil. No text, no logo."},
]

# --- Modèles (révisions Hugging Face figées ; tailles et SHA-256 relevés sur le Hub) ---
MODELS = {
    "high": {
        "role": "Wan2.2 I2V A14B, expert high noise, GGUF Q4_K_M",
        "repo": "bullerwins/Wan2.2-I2V-A14B-GGUF",
        "revision": "c95ab6c210a60ff915aa3f7cb0fa07300b0b2f36",
        "path": "wan2.2_i2v_high_noise_14B_Q4_K_M.gguf",
        "folder": "diffusion_models",
        "size": 9651728896,
        "sha256": "0b7dad1fc7a3f362b306e204537a09648740361c15634f5697d423f55e229573",
        "license": "Apache-2.0 (dépôt GGUF et modèle d'origine Wan-AI/Wan2.2-I2V-A14B)",
    },
    "low": {
        "role": "Wan2.2 I2V A14B, expert low noise, GGUF Q4_K_M",
        "repo": "bullerwins/Wan2.2-I2V-A14B-GGUF",
        "revision": "c95ab6c210a60ff915aa3f7cb0fa07300b0b2f36",
        "path": "wan2.2_i2v_low_noise_14B_Q4_K_M.gguf",
        "folder": "diffusion_models",
        "size": 9651728896,
        "sha256": "0c170a4ae4228a7d30a2cb4dded6eb2bc48ccb1ce761445bb092ff7c5dd4f47b",
        "license": "Apache-2.0 (dépôt GGUF et modèle d'origine Wan-AI/Wan2.2-I2V-A14B)",
    },
    "text_encoder": {
        "role": "Encodeur de texte umt5-xxl, fp8 e4m3fn scaled",
        "repo": "Comfy-Org/Wan_2.1_ComfyUI_repackaged",
        "revision": "123acf1cc74bccbb9bfff8ac1ee72edc08c2341d",
        "path": "split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
        "folder": "text_encoders",
        "size": 6735906897,
        "sha256": "c3355d30191f1f066b26d93fba017ae9809dce6c627dda5f6a66eaa651204f68",
        "license": "Apache-2.0",
    },
    "vae": {
        "role": "VAE Wan 2.1 (utilisé par Wan2.2 A14B)",
        "repo": "Comfy-Org/Wan_2.1_ComfyUI_repackaged",
        "revision": "123acf1cc74bccbb9bfff8ac1ee72edc08c2341d",
        "path": "split_files/vae/wan_2.1_vae.safetensors",
        "folder": "vae",
        "size": 253815318,
        "sha256": "2fc39d31359a4b0a64f55876d8ff7fa8d780956ae2cb13463b0223e15148976b",
        "license": "Apache-2.0",
    },
    "lora_high": {
        "role": "LoRA de vitesse Lightning 4 pas (250928), high noise, rang 128, fp16",
        "repo": "Kijai/WanVideo_comfy",
        "revision": "8260d429d19fd7a72304cad059160b95d843913f",
        "path": "LoRAs/Wan22-Lightning/Wan22_A14B_T2V_HIGH_Lightning_4steps_lora_250928_rank128_fp16.safetensors",
        "folder": "loras",
        "size": 1226980192,
        "sha256": "c000d33fab854def3915557ae5b0cfe6d8993b985c24176f667043d4e7934fdc",
        "license": "Apache-2.0 (original lightx2v/Wan2.2-Lightning) ; conversion fp16 Kijai (dépôt sans licence déclarée)",
    },
    "lora_low": {
        "role": "LoRA de vitesse Lightning 4 pas (250928), low noise, rang 64, fp16",
        "repo": "Kijai/WanVideo_comfy",
        "revision": "8260d429d19fd7a72304cad059160b95d843913f",
        "path": "LoRAs/Wan22-Lightning/Wan22_A14B_T2V_LOW_Lightning_4steps_lora_250928_rank64_fp16.safetensors",
        "folder": "loras",
        "size": 613561776,
        "sha256": "a0f1937980c3c640860b8a58698fdaf7c487af4542d546d692231a3b9d230541",
        "license": "Apache-2.0 (original lightx2v/Wan2.2-Lightning) ; conversion fp16 Kijai (dépôt sans licence déclarée)",
    },
}
for _m in MODELS.values():
    _m["filename"] = _m["path"].split("/")[-1]
    _m["url"] = f"https://huggingface.co/{_m['repo']}/resolve/{_m['revision']}/{_m['path']}"
VERIFY_SHA256 = False                # True : recalcule le SHA-256 de chaque modèle (~2-4 min de plus)

# --- Logiciels (versions figées) ---
PYTHON_SUPPORTED = [(3, 10), (3, 11), (3, 12), (3, 13)]
PIP_TORCH = ["torch==2.6.0", "torchvision==0.21.0", "torchaudio==2.6.0"]
TORCH_INDEX_URL = "https://download.pytorch.org/whl/cu124"
PIP_EXTRA = ["sageattention==1.0.6", "gguf==0.17.1", "imageio-ffmpeg==0.6.0"]
REPOS = {
    "ComfyUI":         {"url": "https://github.com/Comfy-Org/ComfyUI",     "ref": "v0.3.48",
                        "commit": "bff60b5cfc10d1b037a95746226ac6698dc3e373", "license": "GPL-3.0"},
    "ComfyUI_GGUF":    {"url": "https://github.com/city96/ComfyUI-GGUF",   "ref": "cf05733 (2025-08-05)",
                        "commit": "cf0573351ac260d629d460d97f09b09ac17d3726", "license": "Apache-2.0"},
    "ComfyUI_KJNodes": {"url": "https://github.com/kijai/ComfyUI-KJNodes", "ref": "a6b867b (2025-07-28)",
                        "commit": "a6b867b63a29ca48ddb15c589e17a9f2d8530d57", "license": "GPL-3.0"},
}

# --- Contrôles de cohérence ---
assert WIDTH % 16 == 0 and HEIGHT % 16 == 0, "WIDTH et HEIGHT doivent être des multiples de 16"
assert (FRAMES - 1) % 4 == 0, "FRAMES doit valoir 4k+1 (49, 65, 81...)"
assert 0 < HIGH_NOISE_STEPS < STEPS, "il faut 0 < HIGH_NOISE_STEPS < STEPS"
assert SAGE_ATTENTION in ("auto", "off"), "SAGE_ATTENTION doit valoir 'auto' ou 'off'"
for _v in VIDEOS:
    assert _v["start"] in INPUT_IMAGES and _v["end"] in INPUT_IMAGES, f"image inconnue dans {_v['name']}"
print(f"CONFIG chargée : {len(VIDEOS)} vidéos, {WIDTH}x{HEIGHT}, {FRAMES} images à {FPS} i/s "
      f"({FRAMES / FPS:.2f} s), graine {SEED}, {STEPS} pas dont {HIGH_NOISE_STEPS} high noise.")

In [ ]:
# Outils communs (affichage, commandes shell avec sortie visible, journal)
import sys, re, io, gc, json, time, glob, shlex, shutil, hashlib, platform, datetime
import subprocess, collections, traceback, importlib, importlib.util, importlib.metadata, urllib.request

T_START = time.time()
TIMINGS = {}                                     # durées globales (s)
VIDEO_TIMINGS = {v["name"]: {} for v in VIDEOS}  # durées par vidéo (s)

def now_utc():
    return datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")

JOURNAL = {"notebook": "wan22_flf2v_kaggle.ipynb", "debut_utc": now_utc()}

def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

def step(num, title):
    bar = "=" * 78
    print(f"\n{bar}\nÉTAPE {num} — {title}\n{bar}", flush=True)

def gib(n):
    return f"{n / 2**30:.2f} Gio"

def disk_free(path):
    return shutil.disk_usage(path).free

def run(cmd, cwd=None, env=None):
    """Lance une commande, affiche sa sortie en direct, lève une exception si le code retour est non nul."""
    cmd = [str(c) for c in cmd]
    print("$ " + " ".join(shlex.quote(c) for c in cmd), flush=True)
    proc = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, errors="replace", bufsize=1)
    tail = collections.deque(maxlen=60)
    for line in proc.stdout:
        print(line, end="", flush=True)
        tail.append(line)
    code = proc.wait()
    if code != 0:
        raise RuntimeError(f"Échec de la commande (code {code}) : {' '.join(cmd)}\n"
                           f"--- dernières lignes ---\n{''.join(tail)}")

def sha256_file(path, chunk=16 * 2**20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

log("Outils prêts.")

In [ ]:
step(1, "Environnement : GPU, Python, disque, mémoire, Internet")
t0 = time.time()
if not os.path.isdir(OUT_DIR):
    raise RuntimeError(f"{OUT_DIR} introuvable : ce notebook est prévu pour Kaggle.")

log(f"Python {platform.python_version()} ({sys.executable}) sur {platform.platform()}")
if tuple(sys.version_info[:2]) not in PYTHON_SUPPORTED:
    raise RuntimeError(f"Python {platform.python_version()} non pris en charge par torch 2.6.0 "
                       f"(versions acceptées : {PYTHON_SUPPORTED}).")

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("Aucun GPU détecté (nvidia-smi introuvable). Dans Kaggle : Settings > Accelerator > "
                       "GPU T4 x2, puis relancez.")
run(["nvidia-smi"])
gpu_csv = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total,driver_version",
                          "--format=csv,noheader"], capture_output=True, text=True, check=True).stdout.strip()
log("GPU détecté(s) : " + " | ".join(gpu_csv.splitlines()))
log("CUDA_VISIBLE_DEVICES=0 : seul le premier GPU sera utilisé.")
JOURNAL["gpu_nvidia_smi"] = gpu_csv.splitlines()

meminfo = {}
with open("/proc/meminfo") as f:
    for line in f:
        key, val = line.split(":", 1)
        meminfo[key] = int(val.strip().split()[0]) * 1024
log(f"Mémoire vive : {gib(meminfo['MemTotal'])} au total, {gib(meminfo['MemAvailable'])} disponibles")
JOURNAL["ram_totale_octets"] = meminfo["MemTotal"]

existing = [c for c in WORK_DIR_CANDIDATES if os.path.isdir(c)]
if not existing:
    raise RuntimeError(f"Aucun des dossiers {WORK_DIR_CANDIDATES} n'existe.")
print("Espace disque :")
run(["df", "-h", OUT_DIR, *existing])
best = max(existing, key=lambda c: (disk_free(c), -existing.index(c)))
WORK_DIR = os.path.join(best, WORK_SUBDIR)
if os.path.realpath(WORK_DIR).startswith(os.path.realpath(OUT_DIR)):
    raise RuntimeError("Le dossier de travail ne doit pas être dans /kaggle/working.")
os.makedirs(WORK_DIR, exist_ok=True)
MODELS_TOTAL = sum(m["size"] for m in MODELS.values())
log(f"Dossier de travail retenu : {WORK_DIR} ({gib(disk_free(WORK_DIR))} libres)")
log(f"Modèles à prévoir : {gib(MODELS_TOTAL)} ; libre dans {OUT_DIR} : {gib(disk_free(OUT_DIR))}")
JOURNAL["dossier_travail"] = WORK_DIR

probe = "https://huggingface.co/api/models/" + MODELS["high"]["repo"]
log(f"Test d'accès Internet : {probe}")
try:
    with urllib.request.urlopen(urllib.request.Request(probe, headers={"User-Agent": "wan22-flf2v-kaggle"}),
                                timeout=30) as r:
        status = r.status
except Exception as e:
    raise RuntimeError("Pas d'accès à huggingface.co : activez Settings > Internet (On) dans Kaggle "
                       "(compte vérifié par téléphone requis), puis relancez.") from e
if status != 200:
    raise RuntimeError(f"huggingface.co répond HTTP {status}.")
log("Internet : OK")
TIMINGS["environnement_s"] = round(time.time() - t0, 1)

In [ ]:
step(2, f"Images d'entrée : téléchargement puis redimensionnement/recadrage centré en {WIDTH}x{HEIGHT}")
t0 = time.time()
from PIL import Image, ImageOps
from IPython.display import display

INPUT_DIR = os.path.join(WORK_DIR, "inputs")
os.makedirs(INPUT_DIR, exist_ok=True)
UA = "Mozilla/5.0 (X11; Linux x86_64) wan22-flf2v-kaggle"
INPUTS_META, INPUT_PIL = {}, {}

def fetch_image_bytes(url):
    req = urllib.request.Request(url, headers={"User-Agent": UA})
    with urllib.request.urlopen(req, timeout=90) as r:
        status, ctype, data = r.status, r.headers.get("Content-Type", ""), r.read()
    if status != 200:
        raise RuntimeError(f"HTTP {status}")
    if not ctype.startswith("image/"):
        raise RuntimeError(f"la réponse n'est pas une image (Content-Type={ctype!r}, {len(data)} octets)")
    return data, ctype

def load_input(name, url):
    log(f"{name} : téléchargement de {url}")
    try:
        data, ctype = fetch_image_bytes(url)
        source = url
    except Exception as e:
        msg = (f"L'image {name} n'a pas pu être téléchargée ({type(e).__name__}: {e}).\nURL : {url}\n"
               "Ces liens Gradio (…hf.space/…/gradio_api/file=/tmp/gradio/…) sont temporaires : ils expirent "
               "quand le Space redémarre. Régénérez l'image et mettez à jour INPUT_IMAGES dans la cellule CONFIG, "
               f"ou ajoutez un Dataset Kaggle contenant {name}.webp / {name}.png / {name}.jpg et passez "
               "ALLOW_LOCAL_IMAGE_FALLBACK = True.")
        if not ALLOW_LOCAL_IMAGE_FALLBACK:
            raise RuntimeError(msg) from e
        print("!!! " + msg, flush=True)
        found = [p for p in sorted(glob.glob(f"/kaggle/input/**/{name}.*", recursive=True))
                 if p.lower().endswith((".webp", ".png", ".jpg", ".jpeg"))]
        if not found:
            raise RuntimeError(msg + f"\nAucun fichier {name}.* dans /kaggle/input.") from e
        source, ctype = found[0], "fichier local"
        with open(source, "rb") as f:
            data = f.read()
        log(f"{name} : repli sur le fichier local {source}")
    img = Image.open(io.BytesIO(data))
    img.load()
    fmt = img.format
    img = ImageOps.exif_transpose(img).convert("RGB")
    w, h = img.size
    scale = max(WIDTH / w, HEIGHT / h)
    nw, nh = max(WIDTH, round(w * scale)), max(HEIGHT, round(h * scale))
    resized = img.resize((nw, nh), Image.Resampling.LANCZOS)
    left, top = (nw - WIDTH) // 2, (nh - HEIGHT) // 2
    out = resized.crop((left, top, left + WIDTH, top + HEIGHT))
    cropped_pct = 100.0 * (1 - (WIDTH * HEIGHT) / (nw * nh))
    path = os.path.join(INPUT_DIR, f"{name}_{WIDTH}x{HEIGHT}.png")
    out.save(path)
    log(f"{name} : {fmt} {w}x{h} -> redimensionnée {nw}x{nh} -> recadrée au centre {WIDTH}x{HEIGHT} "
        f"({cropped_pct:.1f} % rogné) -> {path}")
    if cropped_pct > 15:
        print(f"!!! {name} : plus de 15 % de l'image est rogné (format d'origine éloigné du 9:16).", flush=True)
    INPUTS_META[name] = {"source": source, "content_type": ctype, "octets": len(data),
                         "sha256": hashlib.sha256(data).hexdigest(), "format": fmt,
                         "taille_origine": [w, h], "taille_redimensionnee": [nw, nh],
                         "taille_finale": [WIDTH, HEIGHT], "pourcentage_rogne": round(cropped_pct, 2)}
    return out

for name, url in INPUT_IMAGES.items():
    INPUT_PIL[name] = load_input(name, url)

# Planche de contrôle (affichée seulement, rien n'est écrit dans /kaggle/working)
thumb_w, thumb_h = WIDTH // 2, HEIGHT // 2
sheet = Image.new("RGB", (thumb_w * len(INPUT_PIL), thumb_h), "black")
for i, im in enumerate(INPUT_PIL.values()):
    sheet.paste(im.resize((thumb_w, thumb_h), Image.Resampling.LANCZOS), (i * thumb_w, 0))
print("Images préparées (de gauche à droite) : " + ", ".join(INPUT_PIL))
display(sheet)
TIMINGS["images_s"] = round(time.time() - t0, 1)

In [ ]:
step(3, "Installation : PyTorch 2.6.0 (CUDA 12.4), ComfyUI v0.3.48, ComfyUI-GGUF, KJNodes, outils")
t0 = time.time()
PY = sys.executable
PIP = [PY, "-m", "pip", "install", "--no-cache-dir", "--progress-bar", "off", "--disable-pip-version-check", "-q"]
CONSTRAINTS = os.path.join(WORK_DIR, "constraints.txt")
with open(CONSTRAINTS, "w") as f:
    f.write("\n".join(PIP_TORCH) + "\n")

log("pip : PyTorch figé (rien à faire s'il est déjà présent, sinon ~2,5 Go à télécharger)")
run(PIP + PIP_TORCH + ["--index-url", TORCH_INDEX_URL])
log("Vérification de PyTorch dans un processus séparé")
run([PY, "-c",
     "import torch, torchvision, torchaudio; "
     "print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| torchvision', torchvision.__version__, "
     "'| torchaudio', torchaudio.__version__); "
     "assert torch.cuda.is_available(), 'CUDA indisponible pour torch'; "
     "print('GPU vu par torch :', torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))"])

COMFY_DIR = os.path.join(WORK_DIR, "ComfyUI")

def git_pin(name, dest):
    spec = REPOS[name]
    if not os.path.isdir(os.path.join(dest, ".git")):
        log(f"git : clonage de {spec['url']}")
        run(["git", "clone", "--quiet", "--filter=blob:none", "--no-checkout", spec["url"], dest])
    run(["git", "-C", dest, "checkout", "--quiet", "--force", spec["commit"]])
    head = subprocess.run(["git", "-C", dest, "rev-parse", "HEAD"],
                          capture_output=True, text=True, check=True).stdout.strip()
    if head != spec["commit"]:
        raise RuntimeError(f"{name} : HEAD {head} au lieu de {spec['commit']}")
    log(f"{name} figé sur {spec['ref']} ({head[:12]}) dans {dest}")

git_pin("ComfyUI", COMFY_DIR)
git_pin("ComfyUI_GGUF", os.path.join(COMFY_DIR, "custom_nodes", "ComfyUI_GGUF"))
git_pin("ComfyUI_KJNodes", os.path.join(COMFY_DIR, "custom_nodes", "ComfyUI_KJNodes"))

# Dépendances ComfyUI, sans torch (déjà figé) ni les paquets de l'interface web (inutiles ici).
SKIP_REQS = {"torch", "torchvision", "torchaudio",
             "comfyui-frontend-package", "comfyui-workflow-templates", "comfyui-embedded-docs"}
reqs = []
with open(os.path.join(COMFY_DIR, "requirements.txt"), encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        pkg = re.split(r"[<>=~!;\[ ]", line, maxsplit=1)[0].strip().lower()
        if pkg not in SKIP_REQS:
            reqs.append(line)
log("pip : dépendances ComfyUI + outils : " + ", ".join(reqs + PIP_EXTRA))
run(PIP + ["-c", CONSTRAINTS] + reqs + PIP_EXTRA)
importlib.invalidate_caches()

import imageio_ffmpeg
FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()
encoders = subprocess.run([FFMPEG, "-hide_banner", "-encoders"], capture_output=True, text=True, check=True).stdout
if "libx264" not in encoders:
    raise RuntimeError(f"ffmpeg ({FFMPEG}) ne propose pas l'encodeur libx264.")
log(f"ffmpeg avec libx264 : {FFMPEG}")

if shutil.which("aria2c") is None:
    log("aria2c absent : installation par apt-get (téléchargements parallèles)")
    try:
        apt_env = dict(os.environ, DEBIAN_FRONTEND="noninteractive")
        run(["apt-get", "update", "-qq"], env=apt_env)
        run(["apt-get", "install", "-y", "-qq", "aria2"], env=apt_env)
    except RuntimeError as e:
        print("!!! Installation d'aria2 impossible : les modèles seront téléchargés avec curl (plus lent).\n"
              + str(e), flush=True)
DOWNLOADER = "aria2c" if shutil.which("aria2c") else "curl"
if DOWNLOADER == "curl" and shutil.which("curl") is None:
    raise RuntimeError("Ni aria2c ni curl ne sont disponibles pour télécharger les modèles.")
log(f"Outil de téléchargement : {DOWNLOADER}")

PKG_VERSIONS = {}
for pkg in ["torch", "torchvision", "torchaudio", "triton", "sageattention", "gguf", "imageio-ffmpeg",
            "numpy", "pillow", "transformers", "safetensors", "einops", "av", "sentencepiece"]:
    try:
        PKG_VERSIONS[pkg] = importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError:
        PKG_VERSIONS[pkg] = None
log("Versions installées : " + ", ".join(f"{k}={v}" for k, v in PKG_VERSIONS.items()))
log(f"Espace libre après installation : {gib(disk_free(WORK_DIR))} dans {WORK_DIR}")
TIMINGS["installation_s"] = round(time.time() - t0, 1)

In [ ]:
step(4, "Modèles : téléchargement (taille vérifiée à l'octet près) hors de /kaggle/working")
t0 = time.time()
MODELS_DIR = os.path.join(COMFY_DIR, "models")

def model_path(key):
    m = MODELS[key]
    return os.path.join(MODELS_DIR, m["folder"], m["filename"])

def model_complete(key):
    p = model_path(key)
    return (os.path.isfile(p) and not os.path.exists(p + ".aria2")
            and os.path.getsize(p) == MODELS[key]["size"])

def download_model(key):
    m, dest = MODELS[key], model_path(key)
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if model_complete(key):
        log(f"{m['filename']} déjà présent ({gib(m['size'])}) : pas de téléchargement")
        return
    log(f"Téléchargement de {m['filename']} ({gib(m['size'])}) depuis {m['repo']} @ {m['revision'][:10]}")
    t = time.time()
    if DOWNLOADER == "aria2c":
        run(["aria2c", "--console-log-level=warn", "--show-console-readout=false", "--summary-interval=30",
             "-x", "16", "-s", "16", "-k", "1M", "-c", "--file-allocation=none",
             "--auto-file-renaming=false", "--allow-overwrite=true", "--max-tries=10", "--retry-wait=5",
             "--connect-timeout=30", "--timeout=120",
             "-d", os.path.dirname(dest), "-o", m["filename"], m["url"]])
    else:
        run(["curl", "-L", "--fail", "-sS", "--retry", "10", "--retry-delay", "5", "-C", "-",
             "-o", dest, m["url"]])
    size = os.path.getsize(dest) if os.path.isfile(dest) else -1
    if size != m["size"] or os.path.exists(dest + ".aria2"):
        raise RuntimeError(f"{m['filename']} incomplet : {size} octets au lieu de {m['size']}.")
    dt = max(time.time() - t, 1e-6)
    log(f"OK {m['filename']} : {gib(size)} en {dt:.0f} s ({size / 2**20 / dt:.0f} Mio/s)")
    if VERIFY_SHA256:
        log(f"SHA-256 de {m['filename']}…")
        digest = sha256_file(dest)
        if digest != m["sha256"]:
            raise RuntimeError(f"SHA-256 inattendu pour {m['filename']} : {digest}")
        m["sha256_verifie"] = True
        log("SHA-256 conforme")

ORDER = ["vae", "lora_high", "lora_low", "text_encoder", "high", "low"]
margin = 2 * 2**30
free = disk_free(WORK_DIR)
need_all = sum(MODELS[k]["size"] for k in ORDER if not model_complete(k))
need_low_disk = sum(MODELS[k]["size"] for k in ORDER if k != "low" and not model_complete(k))
if free >= need_all + margin:
    LOW_DISK_MODE = False
    log(f"Espace disque suffisant : {gib(free)} libres pour {gib(need_all)} à télécharger")
elif free >= need_low_disk + margin:
    LOW_DISK_MODE = True
    print(f"!!! Espace limité ({gib(free)} libres pour {gib(need_all)}) : mode économe. Le modèle low noise "
          "sera téléchargé après la passe high noise, une fois le modèle high noise supprimé.", flush=True)
else:
    raise RuntimeError(f"Espace disque insuffisant dans {WORK_DIR} : {gib(free)} libres, il faut au moins "
                       f"{gib(need_low_disk + margin)} (ou {gib(need_all + margin)} sans le mode économe).")

for key in ORDER:
    if LOW_DISK_MODE and key == "low":
        log("Modèle low noise : téléchargement différé (mode économe)")
        continue
    download_model(key)

run(["du", "-sh", MODELS_DIR])
log(f"Espace libre restant : {gib(disk_free(WORK_DIR))}")
TIMINGS["telechargements_s"] = round(time.time() - t0, 1)

In [ ]:
step(5, "Chargement de ComfyUI (bibliothèque) et des nœuds figés ; test de SageAttention")
t0 = time.time()
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(name)s | %(message)s", force=True)
for noisy in ("httpx", "httpcore", "urllib3", "PIL", "matplotlib", "numba", "h5py", "asyncio"):
    logging.getLogger(noisy).setLevel(logging.WARNING)

importlib.invalidate_caches()
if COMFY_DIR not in sys.path:
    sys.path.insert(0, COMFY_DIR)

import numpy as np
import torch
import torch.nn.functional as F
log(f"torch {torch.__version__} (CUDA {torch.version.cuda})")
if not torch.cuda.is_available():
    raise RuntimeError("torch ne voit pas de GPU CUDA.")
cap = torch.cuda.get_device_capability(0)
arch = f"sm_{cap[0]}{cap[1]}"
if arch not in torch.cuda.get_arch_list():
    raise RuntimeError(f"Le GPU {torch.cuda.get_device_name(0)} ({arch}) n'est pas pris en charge par ce build "
                       f"de torch ({torch.cuda.get_arch_list()}).")
props = torch.cuda.get_device_properties(0)
log(f"GPU utilisé : {props.name}, {arch}, {gib(props.total_memory)} de VRAM")
JOURNAL["gpu"] = {"nom": props.name, "capacite": arch, "vram_octets": props.total_memory}

# Options ComfyUI à poser AVANT d'importer comfy.model_management :
# pas de xformers (Kaggle peut en fournir une version compilée pour un autre torch), attention PyTorch.
import comfy.options
import comfy.cli_args
comfy.cli_args.args.disable_xformers = True
comfy.cli_args.args.use_pytorch_cross_attention = True

import comfy.model_management as mm
import comfyui_version
import nodes as comfy_nodes                                   # CLIPLoader, CLIPTextEncode, VAELoader, VAEDecode,
from comfy_extras.nodes_wan import WanFirstLastFrameToVideo   # LoraLoaderModelOnly, KSamplerAdvanced
from comfy_extras.nodes_model_advanced import ModelSamplingSD3
from custom_nodes.ComfyUI_GGUF.nodes import UnetLoaderGGUF

# KJNodes : on charge uniquement le module des optimisations (NAG, SageAttention) à partir de son fichier,
# sans exécuter le __init__ du paquet (qui importe le serveur web de ComfyUI).
kj_file = os.path.join(COMFY_DIR, "custom_nodes", "ComfyUI_KJNodes", "nodes", "model_optimization_nodes.py")
kj_spec = importlib.util.spec_from_file_location("kjnodes_model_optimization_nodes", kj_file)
kjopt = importlib.util.module_from_spec(kj_spec)
sys.modules[kj_spec.name] = kjopt
kj_spec.loader.exec_module(kjopt)
WanVideoNAG = kjopt.WanVideoNAG
PathchSageAttentionKJ = kjopt.PathchSageAttentionKJ

log(f"ComfyUI {comfyui_version.__version__} importé depuis {COMFY_DIR}")
log(f"Périphérique ComfyUI : {mm.get_torch_device_name(mm.get_torch_device())} | "
    f"mode VRAM : {mm.vram_state.name} | dtype de calcul par défaut : {mm.unet_dtype()}")

def free_vram(label=""):
    mm.unload_all_models()
    gc.collect()
    mm.soft_empty_cache()
    torch.cuda.empty_cache()
    free_b, total_b = torch.cuda.mem_get_info()
    log(f"VRAM libre {label}: {gib(free_b)} / {gib(total_b)}")

# --- Test de SageAttention (noyaux Triton) : activée seulement si elle tourne ET donne le même résultat ---
USE_SAGE = False
SAGE_REPORT = {"demande": SAGE_ATTENTION}
if SAGE_ATTENTION == "off":
    log("SageAttention désactivée dans la CONFIG : attention PyTorch standard.")
else:
    log("Test de SageAttention sur ce GPU (comparaison avec l'attention PyTorch)…")
    try:
        from sageattention import sageattn
        gen = torch.Generator(device="cuda").manual_seed(0)
        worst, finite = 1.0, True
        for lq, lk in [(4096, 4096), (4096, 512)]:          # auto-attention et attention croisée (texte)
            q = torch.randn(1, lq, 40, 128, device="cuda", dtype=torch.float16, generator=gen)
            k = torch.randn(1, lk, 40, 128, device="cuda", dtype=torch.float16, generator=gen)
            v = torch.randn(1, lk, 40, 128, device="cuda", dtype=torch.float16, generator=gen)
            out = sageattn(q, k.clone(), v, tensor_layout="NHD", is_causal=False)
            ref = F.scaled_dot_product_attention(q.transpose(1, 2), k.transpose(1, 2),
                                                 v.transpose(1, 2)).transpose(1, 2)
            finite = finite and bool(torch.isfinite(out).all())
            cos = F.cosine_similarity(out.float().flatten(), ref.float().flatten(), dim=0).item()
            worst = min(worst, cos)
            log(f"  q={lq} k={lk} : similarité cosinus {cos:.5f}")
        del q, k, v, out, ref
        torch.cuda.empty_cache()
        USE_SAGE = finite and worst > 0.99
        SAGE_REPORT.update({"similarite_min": round(worst, 6), "valeurs_finies": finite})
        if not USE_SAGE:
            print(f"!!! SageAttention donne un résultat trop différent (cos={worst:.4f}, finies={finite}) : "
                  "attention PyTorch standard (plus lente).", flush=True)
    except Exception as e:
        print("!!! SageAttention indisponible sur ce GPU : attention PyTorch standard (plus lente). Détail :",
              flush=True)
        traceback.print_exc()
        SAGE_REPORT["erreur"] = f"{type(e).__name__}: {e}"
        USE_SAGE = False
SAGE_REPORT["utilisee"] = USE_SAGE
log(f"SageAttention utilisée : {USE_SAGE}")
TIMINGS["chargement_comfyui_s"] = round(time.time() - t0, 1)

In [ ]:
step(6, "Encodage des prompts (umt5-xxl fp8) puis conditionnement première/dernière image (VAE)")

def pil_to_image_tensor(img):
    arr = np.asarray(img.convert("RGB"), dtype=np.float32) / 255.0
    return torch.from_numpy(arr).unsqueeze(0)              # format IMAGE de ComfyUI : [1, H, W, 3], 0..1

IMAGES_T = {name: pil_to_image_tensor(im) for name, im in INPUT_PIL.items()}

t0 = time.time()
with torch.inference_mode():
    log(f"Chargement de l'encodeur de texte {MODELS['text_encoder']['filename']}")
    clip = comfy_nodes.CLIPLoader().load_clip(MODELS["text_encoder"]["filename"], "wan", "default")[0]
    encoder = comfy_nodes.CLIPTextEncode()
    log("Encodage du prompt négatif")
    NEG_COND = encoder.encode(clip, NEGATIVE_PROMPT)[0]
    POS_COND = {}
    for v in VIDEOS:
        log(f"Encodage du prompt de {v['name']}")
        POS_COND[v["name"]] = encoder.encode(clip, v["prompt"])[0]
    del clip
free_vram("après encodage du texte ")
TIMINGS["encodage_texte_s"] = round(time.time() - t0, 1)

t0 = time.time()
log(f"Chargement du VAE {MODELS['vae']['filename']}")
VAE = comfy_nodes.VAELoader().load_vae(MODELS["vae"]["filename"])[0]
TIMINGS["chargement_vae_s"] = round(time.time() - t0, 1)

expected_latent = (1, 16, (FRAMES - 1) // 4 + 1, HEIGHT // 8, WIDTH // 8)
COND = {}
flf = WanFirstLastFrameToVideo()
for v in VIDEOS:
    t = time.time()
    log(f"{v['name']} : première image {v['start']}, dernière image {v['end']} -> encodage VAE ({FRAMES} images)")
    with torch.inference_mode():
        pos, neg, latent = flf.encode(positive=POS_COND[v["name"]], negative=NEG_COND, vae=VAE,
                                      width=WIDTH, height=HEIGHT, length=FRAMES, batch_size=1,
                                      start_image=IMAGES_T[v["start"]], end_image=IMAGES_T[v["end"]])
    shape = tuple(latent["samples"].shape)
    if shape != expected_latent:
        raise RuntimeError(f"{v['name']} : latent de forme {shape}, attendu {expected_latent}")
    COND[v["name"]] = (pos, neg, latent)
    VIDEO_TIMINGS[v["name"]]["conditionnement_s"] = round(time.time() - t, 1)
    log(f"{v['name']} : conditionnement prêt, latent {shape} ({VIDEO_TIMINGS[v['name']]['conditionnement_s']} s)")
free_vram("après conditionnement ")

In [ ]:
step(7, f"Passe high noise : pas 0 → {HIGH_NOISE_STEPS} sur {STEPS} pour " + ", ".join(v["name"] for v in VIDEOS))

def check_lora(model, key):
    n = len(model.patches)
    log(f"LoRA {MODELS[key]['filename']} : {n} couches modifiées")
    if n < 390:   # la LoRA Lightning touche 400 couches linéaires (40 blocs x 10)
        raise RuntimeError(f"La LoRA {MODELS[key]['filename']} ne s'applique pas ({n} couches au lieu de 400).")

def check_latent(lat, label):
    s = lat["samples"]
    if not bool(torch.isfinite(s).all()):
        raise RuntimeError(f"{label} : latents non finis (NaN/Inf), probablement un dépassement en fp16.")
    log(f"{label} : latents {tuple(s.shape)}, moyenne {s.float().mean().item():.3f}, "
        f"écart-type {s.float().std().item():.3f}")

t0 = time.time()
with torch.inference_mode():
    log(f"Chargement de {MODELS['high']['filename']}")
    model = UnetLoaderGGUF().load_unet(MODELS["high"]["filename"])[0]
    log(f"NAG (échelle {NAG_SCALE}, alpha {NAG_ALPHA}, tau {NAG_TAU}) avec le prompt négatif")
    model = WanVideoNAG().patch(model, NEG_COND, NAG_SCALE, NAG_ALPHA, NAG_TAU)[0]
    log(f"Shift {SHIFT_HIGH}")
    model = ModelSamplingSD3().patch(model, SHIFT_HIGH)[0]
    log(f"LoRA high noise (force {LORA_STRENGTH_HIGH})")
    model = comfy_nodes.LoraLoaderModelOnly().load_lora_model_only(model, MODELS["lora_high"]["filename"],
                                                                   LORA_STRENGTH_HIGH)[0]
    check_lora(model, "lora_high")
    if USE_SAGE:
        log("SageAttention activée pour ce modèle")
        model = PathchSageAttentionKJ().patch(model, "auto")[0]
TIMINGS["preparation_high_noise_s"] = round(time.time() - t0, 1)

LAT_HIGH = {}
sampler_node = comfy_nodes.KSamplerAdvanced()
for v in VIDEOS:
    t = time.time()
    pos, neg, latent = COND[v["name"]]
    log(f"{v['name']} : échantillonnage high noise ({SAMPLER}/{SCHEDULER}, CFG {CFG}, graine {SEED})")
    with torch.inference_mode():
        LAT_HIGH[v["name"]] = sampler_node.sample(
            model=model, add_noise="enable", noise_seed=SEED, steps=STEPS, cfg=CFG,
            sampler_name=SAMPLER, scheduler=SCHEDULER, positive=pos, negative=neg, latent_image=latent,
            start_at_step=0, end_at_step=HIGH_NOISE_STEPS, return_with_leftover_noise="enable")[0]
    check_latent(LAT_HIGH[v["name"]], f"{v['name']} high noise")
    VIDEO_TIMINGS[v["name"]]["high_noise_s"] = round(time.time() - t, 1)
    log(f"{v['name']} : passe high noise terminée en {VIDEO_TIMINGS[v['name']]['high_noise_s']} s")

del model
free_vram("après la passe high noise ")
if LOW_DISK_MODE:
    os.remove(model_path("high"))
    log("Mode économe : modèle high noise supprimé du disque")

In [ ]:
step(8, f"Passe low noise : pas {HIGH_NOISE_STEPS} → {STEPS} pour " + ", ".join(v["name"] for v in VIDEOS))
if LOW_DISK_MODE:
    t = time.time()
    download_model("low")
    TIMINGS["telechargement_low_differe_s"] = round(time.time() - t, 1)

t0 = time.time()
with torch.inference_mode():
    log(f"Chargement de {MODELS['low']['filename']}")
    model = UnetLoaderGGUF().load_unet(MODELS["low"]["filename"])[0]
    log(f"Shift {SHIFT_LOW}")
    model = ModelSamplingSD3().patch(model, SHIFT_LOW)[0]
    log(f"LoRA low noise (force {LORA_STRENGTH_LOW})")
    model = comfy_nodes.LoraLoaderModelOnly().load_lora_model_only(model, MODELS["lora_low"]["filename"],
                                                                   LORA_STRENGTH_LOW)[0]
    check_lora(model, "lora_low")
    if USE_SAGE:
        log("SageAttention activée pour ce modèle")
        model = PathchSageAttentionKJ().patch(model, "auto")[0]
TIMINGS["preparation_low_noise_s"] = round(time.time() - t0, 1)

LAT_LOW = {}
for v in VIDEOS:
    t = time.time()
    pos, neg, _ = COND[v["name"]]
    log(f"{v['name']} : échantillonnage low noise")
    with torch.inference_mode():
        LAT_LOW[v["name"]] = sampler_node.sample(
            model=model, add_noise="disable", noise_seed=SEED, steps=STEPS, cfg=CFG,
            sampler_name=SAMPLER, scheduler=SCHEDULER, positive=pos, negative=neg,
            latent_image=LAT_HIGH[v["name"]], start_at_step=HIGH_NOISE_STEPS, end_at_step=10000,
            return_with_leftover_noise="disable")[0]
    check_latent(LAT_LOW[v["name"]], f"{v['name']} low noise")
    VIDEO_TIMINGS[v["name"]]["low_noise_s"] = round(time.time() - t, 1)
    log(f"{v['name']} : passe low noise terminée en {VIDEO_TIMINGS[v['name']]['low_noise_s']} s")

del model
free_vram("après la passe low noise ")

In [ ]:
step(9, "Décodage VAE puis encodage MP4 (H.264, yuv420p) dans /kaggle/working")

def probe_video(path):
    r = subprocess.run([FFMPEG, "-hide_banner", "-i", path, "-map", "0:v:0", "-f", "null", "-"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"Lecture de contrôle impossible pour {path} :\n{r.stderr[-3000:]}")
    frames = re.findall(r"frame=\s*(\d+)", r.stderr)
    stream = re.search(r"Video: (\w+)[^,]*, (\w+)(?:\([^)]*\))?, (\d+)x(\d+)", r.stderr)
    if not frames or not stream:
        raise RuntimeError(f"Sortie ffmpeg inattendue pour {path} :\n{r.stderr[-3000:]}")
    return {"images": int(frames[-1]), "codec": stream.group(1), "pix_fmt": stream.group(2),
            "largeur": int(stream.group(3)), "hauteur": int(stream.group(4))}

OUTPUTS_META, PREVIEW_ROWS = {}, []
for v in VIDEOS:
    name = v["name"]
    t = time.time()
    log(f"{name} : décodage VAE")
    with torch.inference_mode():
        images = comfy_nodes.VAEDecode().decode(VAE, LAT_LOW[name])[0]
    frames = images.clamp(0, 1).mul(255).round().to(torch.uint8).cpu().numpy()
    del images
    if frames.shape != (FRAMES, HEIGHT, WIDTH, 3):
        raise RuntimeError(f"{name} : images décodées de forme {frames.shape}, attendu {(FRAMES, HEIGHT, WIDTH, 3)}")
    VIDEO_TIMINGS[name]["decodage_vae_s"] = round(time.time() - t, 1)
    log(f"{name} : {frames.shape[0]} images décodées (moyenne {frames.mean():.1f}, écart-type {frames.std():.1f})")
    if frames.std() < 2:
        print(f"!!! {name} : images quasi uniformes (vidéo noire ou grise ?) — vérifiez le résultat.", flush=True)

    t = time.time()
    tmp_path = os.path.join(WORK_DIR, f"{name}.tmp.mp4")
    out_path = os.path.join(OUT_DIR, f"{name}.mp4")
    cmd = [FFMPEG, "-y", "-hide_banner", "-loglevel", "error",
           "-f", "rawvideo", "-pix_fmt", "rgb24", "-s", f"{WIDTH}x{HEIGHT}", "-r", str(FPS), "-i", "-",
           "-an", "-c:v", "libx264", "-preset", X264_PRESET, "-crf", str(X264_CRF), "-pix_fmt", "yuv420p",
           "-movflags", "+faststart", tmp_path]
    log(f"{name} : encodage H.264 ({' '.join(shlex.quote(c) for c in cmd[1:])})")
    r = subprocess.run(cmd, input=frames.tobytes(), capture_output=True)
    if r.returncode != 0:
        raise RuntimeError(f"ffmpeg a échoué pour {name} :\n{r.stderr.decode(errors='replace')[-3000:]}")
    shutil.move(tmp_path, out_path)
    info = probe_video(out_path)
    if (info["images"], info["codec"], info["pix_fmt"], info["largeur"], info["hauteur"]) != \
            (FRAMES, "h264", "yuv420p", WIDTH, HEIGHT):
        raise RuntimeError(f"{name} : vidéo non conforme : {info}")
    VIDEO_TIMINGS[name]["encodage_mp4_s"] = round(time.time() - t, 1)
    info.update({"fichier": out_path, "octets": os.path.getsize(out_path), "fps": FPS,
                 "duree_s": round(FRAMES / FPS, 3)})
    OUTPUTS_META[name] = info
    log(f"{name} : {out_path} ({info['octets'] / 2**20:.2f} Mio, {info['images']} images, {info['codec']}, "
        f"{info['pix_fmt']}, {info['largeur']}x{info['hauteur']})")
    PREVIEW_ROWS.append([INPUT_PIL[v["start"]]] + [Image.fromarray(frames[i]) for i in
                        (0, FRAMES // 2, FRAMES - 1)] + [INPUT_PIL[v["end"]]])
    del frames

# Planche de contrôle affichée (non enregistrée) : image de départ | images 1, milieu, dernière | image de fin
tw, th = WIDTH // 3, HEIGHT // 3
sheet = Image.new("RGB", (tw * 5, th * len(PREVIEW_ROWS)), "black")
for r_i, row in enumerate(PREVIEW_ROWS):
    for c_i, im in enumerate(row):
        sheet.paste(im.resize((tw, th), Image.Resampling.LANCZOS), (c_i * tw, r_i * th))
print("Contrôle par ligne (" + ", ".join(v["name"] for v in VIDEOS) +
      ") : entrée début | image 1 | image du milieu | dernière image | entrée fin")
display(sheet)
del VAE
free_vram("après décodage ")

In [ ]:
step(10, "Journal (journal.json) et nettoyage")
for name in VIDEO_TIMINGS:
    VIDEO_TIMINGS[name]["total_s"] = round(sum(VIDEO_TIMINGS[name].values()), 1)

JOURNAL.update({
    "fin_utc": now_utc(),
    "duree_totale_notebook_s": round(time.time() - T_START, 1),
    "parametres": {
        "largeur": WIDTH, "hauteur": HEIGHT, "images": FRAMES, "fps": FPS, "duree_s": round(FRAMES / FPS, 3),
        "graine": SEED, "pas": STEPS, "pas_high_noise": HIGH_NOISE_STEPS, "cfg": CFG,
        "sampler": SAMPLER, "scheduler": SCHEDULER, "shift_high": SHIFT_HIGH, "shift_low": SHIFT_LOW,
        "force_lora_high": LORA_STRENGTH_HIGH, "force_lora_low": LORA_STRENGTH_LOW,
        "nag": {"echelle": NAG_SCALE, "alpha": NAG_ALPHA, "tau": NAG_TAU, "applique_a": "high noise"},
        "sage_attention": SAGE_REPORT, "x264": {"crf": X264_CRF, "preset": X264_PRESET, "pix_fmt": "yuv420p"},
        "mode_disque_econome": LOW_DISK_MODE,
    },
    "prompts": {"negatif": NEGATIVE_PROMPT, "videos": VIDEOS},
    "images_entree": INPUTS_META,
    "modeles": {k: {"role": m["role"], "depot": m["repo"], "revision": m["revision"], "fichier": m["path"],
                    "url": m["url"], "octets": m["size"], "sha256_hub": m["sha256"],
                    "sha256_verifie": m.get("sha256_verifie", False), "licence": m["license"]}
                for k, m in MODELS.items()},
    "logiciels": {"python": platform.python_version(), "paquets": PKG_VERSIONS,
                  "comfyui_version": comfyui_version.__version__,
                  "depots": {k: {"url": r["url"], "ref": r["ref"], "commit": r["commit"], "licence": r["license"]}
                             for k, r in REPOS.items()},
                  "ffmpeg": FFMPEG},
    "licences": {
        "Wan2.2-I2V-A14B (Wan-AI)": "Apache-2.0",
        "GGUF bullerwins/Wan2.2-I2V-A14B-GGUF": "Apache-2.0",
        "umt5-xxl fp8 et VAE Wan 2.1 (Comfy-Org/Wan_2.1_ComfyUI_repackaged)": "Apache-2.0",
        "LoRA Lightning 250928 (lightx2v/Wan2.2-Lightning)": "Apache-2.0 ; fichiers fp16 de Kijai/WanVideo_comfy "
                                                              "(dépôt sans licence déclarée)",
        "ComfyUI": "GPL-3.0", "ComfyUI-GGUF": "Apache-2.0", "ComfyUI-KJNodes": "GPL-3.0",
        "SageAttention 1.0.6": "BSD-3-Clause", "PyTorch": "BSD-3-Clause",
        "ffmpeg (imageio-ffmpeg, libx264)": "GPL (outil d'encodage, non redistribué)",
    },
    "durees_s": TIMINGS,
    "durees_par_video_s": VIDEO_TIMINGS,
    "sorties": OUTPUTS_META,
})
journal_path = os.path.join(OUT_DIR, "journal.json")
with open(journal_path, "w", encoding="utf-8") as f:
    json.dump(JOURNAL, f, ensure_ascii=False, indent=2)
log(f"Journal écrit : {journal_path}")

# Nettoyage : aucun fichier temporaire ne doit rester (ni ici ni dans /kaggle/working)
for p in glob.glob(os.path.join(WORK_DIR, "*.tmp.mp4")):
    os.remove(p)
    log(f"Supprimé : {p}")
if DELETE_WORK_DIR_AT_END:
    shutil.rmtree(WORK_DIR)
    log(f"Dossier de travail supprimé : {WORK_DIR}")

expected = {"V1.mp4", "V2.mp4", "journal.json"}
print(f"\nContenu de {OUT_DIR} :")
for root, dirs, files in os.walk(OUT_DIR):
    for fn in files:
        p = os.path.join(root, fn)
        size = os.path.getsize(p)
        rel = os.path.relpath(p, OUT_DIR)
        flag = "" if rel in expected else ("   <- inattendu, VOLUMINEUX" if size > 10 * 2**20 else "   <- inattendu")
        print(f"  {rel:40s} {size / 2**20:10.2f} Mio{flag}")
missing = [e for e in expected if not os.path.isfile(os.path.join(OUT_DIR, e))]
if missing:
    raise RuntimeError(f"Sorties manquantes : {missing}")

print("\nRésumé des durées (s) :")
for name, d in VIDEO_TIMINGS.items():
    print(f"  {name} : " + ", ".join(f"{k}={val}" for k, val in d.items()))
print("  global : " + ", ".join(f"{k}={val}" for k, val in TIMINGS.items()))
log(f"Terminé en {(time.time() - T_START) / 60:.1f} min. Fichiers : V1.mp4, V2.mp4, journal.json dans {OUT_DIR}")